# Human data

The ancestral alleles most widely used for human data are those of the Ensembl EPO ancestral sequence, which is reconstructed from one reference genome per primate species on a fixed species tree. This guide runs {class}`~ancestree.inference.FixedTreeInference` and {class}`~ancestree.local_tree_inference.LocalTreeInference` on a sample of human genomes and the same primate outgroups and compares their assignments with EPO's. The data cover chr1:95–100 Mb (GRCh38). `msl.vcf.gz` holds 10 phased diploid individuals of the Mende in Sierra Leone (MSL) sample of the 1000 Genomes Project, which form the ingroup of 20 haplotypes, together with the nine primate outgroups of the Ensembl 10-primate EPO alignment as one haploid sample per species, missing where the species does not align. It carries a record wherever the sample or an outgroup has a second base, and its `INFO/AA` field holds the ancestral base of the EPO sequence, in lower case where EPO marks it as less confident. `msl.bed` lists the accessible positions, at which every haplotype of the sample is called and EPO aligns the human sequence in one block and reports an ancestral base. `msl_mu.bedgraph.gz` gives the germline mutation rate at each accessible position from the Roulette model ([Seplyarskiy et al., 2023](https://doi.org/10.1038/s41588-023-01562-0)), scaled to a mean of $1.25 \times 10^{-8}$ per site per generation over these positions, and `msl_rec.txt` is the HapMap II recombination map over the region.

In [1]:
from cyvcf2 import VCF

outgroups = ["chimpanzee", "bonobo", "gorilla", "orangutan", "gibbon", "rhesus",
             "cynomolgus", "vervet", "mouse_lemur"]
ids = [s for s in VCF("msl.vcf.gz").samples if s not in outgroups]
print(f"{len(ids)} ingroup individuals, {len(outgroups)} outgroups")

10 ingroup individuals, 9 outgroups


## Model inputs

Both modes use HKY with the transition/transversion ratio $\kappa$ and the stationary base frequencies $\pi = (\pi_A, \pi_C, \pi_G, \pi_T)$ estimated over the accessible positions of chr1:93.75–100 Mb. The fit of fixed-tree mode is calibrated against the number of accessible positions.

In [2]:
import numpy as np
import ancestree as anc

model = anc.HKY(kappa=4.15)
pi = [0.313, 0.185, 0.185, 0.317]
access = np.loadtxt("msl.bed", usecols=(1, 2), dtype=np.int64)
n_access = int(np.diff(access).sum())
composition = anc.BaseComposition.from_n_target_sites(n_access, stationary=pi)
print(f"{n_access:,} accessible positions")

4,013,074 accessible positions


## Fixed-tree mode

{class}`~ancestree.inference.FixedTreeInference` places one outgroup per nested lineage on an {class}`~ancestree.trees.OutgroupLadderTree`, so the remaining three outgroups are left out, and fits the branch rates of the ladder by maximum likelihood. The posterior is read at the ancestor the ingroup shares with chimpanzee, the first node of the ladder above the ingroup and the node EPO reports (see {doc}`focal_node`).

In [3]:
ladder = ["chimpanzee", "gorilla", "orangutan", "gibbon", "rhesus", "mouse_lemur"]
fixed_tree = anc.Inference.from_fixed_tree(
    "msl.vcf.gz", ingroup_samples=ids, outgroup_samples=ladder, model=model,
    base_composition=composition, n_target_sites=n_access,
    focal=anc.FocalNode("ingroup_mrca", coalescences=1), n_starts=1,
)
fixed_tree.fit()
res_fixed = list(fixed_tree.infer())

INFO:ancestree.CyVCF2Source: Reading variants from msl.vcf.gz (38 haplotype samples)
INFO:ancestree.FixedTreeInference: Fitting 11 branch rate(s) by ML over the config histogram.
INFO:ancestree.FixedTreeInference: L-BFGS-B converged in 17 iterations (marginal log-likelihood -9725036.0405)
INFO:ancestree.FixedTreeInference: MLE branch rates: K0=5.0389e-03, K1=5.8016e-03, K2=1.4816e-03, K3=7.2223e-03, K4=7.3906e-03, K5=1.4778e-02, K6=2.0932e-03, K7=1.6678e-02, K8=9.8671e-03, K9=2.9731e-02, K10=1.3331e-01
INFO:ancestree.FixedTreeInference: Ingroup MRCA → outgroup divergences: chimpanzee_h0=1.0841e-02, gorilla_h0=1.3743e-02, orangutan_h0=2.8689e-02, gibbon_h0=3.2682e-02, rhesus_h0=5.5602e-02, mouse_lemur_h0=1.5918e-01
INFO:ancestree.FixedTreeInference: Using 10 ingroup sample(s): HG03052, HG03054, HG03055, HG03057, HG03058, HG03060, HG03061, HG03063, HG03064, HG03066
INFO:ancestree.FixedTreeInference: Using 6 outgroup sample(s): chimpanzee_h0, gorilla_h0, orangutan_h0, gibbon_h0, rhesus_h0

## Local-tree mode

{class}`~ancestree.local_tree_inference.LocalTreeInference` reads the accessibility mask ({paramref}`accessibility <ancestree.local_tree_inference.LocalTreeInference.accessibility>`), the mutation map ({paramref}`mutation_map <ancestree.local_tree_inference.LocalTreeInference.mutation_map>`) and the recombination map ({paramref}`recombination_map <ancestree.local_tree_inference.LocalTreeInference.recombination_map>`) from their files, with all nine outgroups as tips, each a haploid sample in a diploid file. The posterior is read at the ingroup MRCA.

In [4]:
local_tree = anc.Inference.from_local_tree(
    "msl.vcf.gz", model, mu=1.25e-8, rec_rate=1e-8, base_composition=composition,
    ingroup_samples=ids, outgroup_samples=outgroups,
    accessibility="msl.bed", mutation_map="msl_mu.bedgraph.gz",
    recombination_map="msl_rec.txt", n_ensemble=64,
)
res_local = list(local_tree.infer())

INFO:ancestree.CyVCF2Source: Reading variants from msl.vcf.gz (38 haplotype samples)
INFO:ancestree.LocalTreeInference: Leaving out 9 haplotype(s) that no call reaches, such as the second slot of a haploid sample in a diploid file: chimpanzee_h1, bonobo_h1, gorilla_h1, orangutan_h1, gibbon_h1, rhesus_h1, cynomolgus_h1, vervet_h1, mouse_lemur_h1
INFO:ancestree.LocalTreeInference: Inferring local trees from genotypes (29 samples, window=8snp)
INFO:ancestree.LocalTreeInference: Using 20 ingroup sample(s): HG03052_h0, HG03052_h1, HG03054_h0, HG03054_h1, HG03055_h0, HG03055_h1, HG03057_h0, HG03057_h1, HG03058_h0, HG03058_h1, HG03060_h0, HG03060_h1, HG03061_h0, HG03061_h1, HG03063_h0, HG03063_h1, HG03064_h0, HG03064_h1, HG03066_h0, HG03066_h1
INFO:ancestree.LocalTreeInference: Using 9 outgroup sample(s): chimpanzee_h0, bonobo_h0, gorilla_h0, orangutan_h0, gibbon_h0, rhesus_h0, cynomolgus_h0, vervet_h0, mouse_lemur_h0
LocalTreeInference: 1 segments [01:20, 80.74s/ segments]
INFO:ancestree.Loc

## Comparison with EPO

The EPO base of each site can take the place of the truth in a {class}`~ancestree.posterior.Grade`, which then reports the fraction of sites whose MAP allele agrees with EPO and the mean Brier score of the posterior against EPO's base, here over the sites polymorphic within the sample ({class}`~ancestree.sites.PolymorphicSiteFilter`).

In [5]:
epo = {record.POS: record.INFO["AA"].upper() for record in VCF("msl.vcf.gz")}
polymorphic = anc.PolymorphicSiteFilter(samples=ids)
grades = {"fixed-tree, human-chimpanzee": anc.Grade(res_fixed, epo, filter=polymorphic),
          "local-tree, ingroup MRCA": anc.Grade(res_local, epo, filter=polymorphic)}
print(f"{'mode, focal node':<30} {'sites':>6} {'MAP agreement':>14} {'mean Brier':>11}")
for label, g in grades.items():
    print(f"{label:<30} {g.n_sites:>6,} {g.map_recovery:>14.1%} {g.brier:>11.4f}")

mode, focal node                sites  MAP agreement  mean Brier
fixed-tree, human-chimpanzee   13,999          99.5%      0.0088
local-tree, ingroup MRCA       13,999          98.0%      0.0353


In [6]:
fixed, local = grades.values()
assert fixed.map_recovery > 0.98
assert local.map_recovery < fixed.map_recovery

Read at the node EPO reports, fixed-tree mode agrees with EPO at nearly every site. Local-tree mode, read at the ingroup MRCA, departs from EPO at more sites. A substitution on the human lineage between the two nodes yields an allele that is derived relative to the human–chimpanzee ancestor but ancestral relative to the ingroup MRCA, the node the unfolded spectrum of a sample refers to.